# Jordan-Wide 6-Model Ensemble — Unified NetCDF Files
## Six Time Periods | SSP2-4.5

**Purpose:**  
Generate six unified Jordan-wide NetCDF files using the **equal-weight 6-model ensemble** — the arithmetic mean of all six available RICCAR CMIP6 models at every grid cell. This represents the traditional multi-model ensemble approach and is the third comparison approach in the manuscript

$$P_{\text{ens6}}(i,j,t) = \frac{1}{6}\sum_{k=1}^{6} M_k(i,j,t)$$

Unlike the basin-specific 3-model ensemble (Notebook 05), all six models contribute equally to every grid cell regardless of basin location. Grid cells outside the Jordan domain are set to `NaN`.

**Six output periods:**

| # | Period label | Description |
|---|-------------|-------------|
| 1 | 1961–2070 | Full simulation period |
| 2 | 1995–2014 | Evaluation / reference period |
| 3 | 2015–2020 | Transitional period |
| 4 | 2021–2040 | Near-future |
| 5 | 2041–2060 | Mid-future |
| 6 | 2061–2070 | End-of-century |

**Output directory:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6 ensemble models\`  
Filename format: `Jordan_6model_ensemble_ssp245_<PERIOD>.nc`


## 1. Import Libraries

In [1]:
import os
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point
import time as _time

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")


Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
SHAPEFILE = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\jordan.basins\surface.basins.for.jordan\Jo.shp"
)

MODEL_DIR = Path(
    r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
    r"\Merge\Precipitation"
)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6 ensemble models"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Models (all 6 — equal weight) ────────────────────────────────────────────
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]
N_MODELS = len(MODELS)

PR_VAR = "prAdjust"

# ── Syria basins to exclude ───────────────────────────────────────────────────
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

# ── Six time periods ──────────────────────────────────────────────────────────
PERIODS = [
    ("1961_2070", 1961, 2070),
    ("1995_2014", 1995, 2014),
    ("2015_2020", 2015, 2020),
    ("2021_2040", 2021, 2040),
    ("2041_2060", 2041, 2060),
    ("2061_2070", 2061, 2070),
]

print("Configuration loaded.")
print(f"  Models     : {N_MODELS} (equal weight — all models contribute to every cell)")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Shapefile  : {SHAPEFILE}")
print(f"  Output dir : {OUTPUT_DIR}")
print()
print("Six output periods:")
for label, y0, y1 in PERIODS:
    print(f"  {label}  ({y0}–{y1})")


Configuration loaded.
  Models     : 6 (equal weight — all models contribute to every cell)
  Model dir  : D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915\Merge\Precipitation
  Shapefile  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\jordan.basins\surface.basins.for.jordan\Jo.shp
  Output dir : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\6 ensemble models

Six output periods:
  1961_2070  (1961–2070)
  1995_2014  (1995–2014)
  2015_2020  (2015–2020)
  2021_2040  (2021–2040)
  2041_2060  (2041–2060)
  2061_2070  (2061–2070)


## 3. Load Basin Shapefile — Jordan Domain Mask

The shapefile is used solely to define the Jordan spatial domain. Syrian portions of transboundary basins are excluded. Unlike the basin-specific 3-model ensemble, no per-basin model assignment is needed — the mask simply identifies which grid cells are inside Jordan.


In [3]:
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy()
gdf_jordan = gdf_jordan.reset_index(drop=True)

print(f"Total shapefile features : {len(gdf_all)}")
print(f"Jordan-only retained     : {len(gdf_jordan)}")
print()
print("Jordan basins included in domain mask:")
for _, row in gdf_jordan.iterrows():
    print(f"  {row['BASIN_NAME']}")


Total shapefile features : 19
Jordan-only retained     : 16

Jordan basins included in domain mask:
  HAMMAD
  YARMOUK (JORDAN)
  J.VALLEY-YARMOUK TRIANGLE
  JORDAN VALLY (JORDAN)
  N.R.S.W
  AZRAQ (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  MUJIB
  D.S.R.S.W
  W. ARABA NORTH
  HASA
  JAFER
  WADI ARABA SOUTH
  QA DISI & SOUTHERN DESERT
  SARHAN


## 4. Build Jordan-Domain Grid Mask and Basin ID

For each grid cell, `shapely.Point.within()` determines whether the cell centre falls inside any Jordan basin polygon. A `basin_id` lookup is also built (same encoding as Notebook 05) so the output NC files carry the same `basin_id` variable — enabling direct comparison between 3-model and 6-model ensemble files in downstream analysis.


In [4]:
# Get grid from first model (all share the same grid)
sample_nc = MODEL_DIR / f"{MODELS[0]}.nc"
with xr.open_dataset(sample_nc) as ds_s:
    lats      = ds_s["lat"].values      # (85,)
    lons      = ds_s["lon"].values      # (75,)
    all_times = ds_s["time"].values     # (40177,)

n_lat = len(lats)
n_lon = len(lons)
jordan_basin_names = gdf_jordan["BASIN_NAME"].tolist()

print(f"Grid      : {n_lat} lat × {n_lon} lon = {n_lat * n_lon:,} cells")
print(f"Full axis : {len(all_times):,} steps  "
      f"({pd.Timestamp(all_times[0]).date()} → {pd.Timestamp(all_times[-1]).date()})")
print()
print("Building Jordan mask and basin_id grid ...")

# (n_lat, n_lon) arrays
jordan_mask   = np.zeros((n_lat, n_lon), dtype=bool)    # True = inside Jordan
basin_id_grid = np.full((n_lat, n_lon), -1, dtype=np.int8)  # -1 = outside

t0 = _time.time()
for li, lat in enumerate(lats):
    for lj, lon in enumerate(lons):
        pt = Point(lon, lat)
        for bi, bname in enumerate(jordan_basin_names):
            if gdf_jordan.iloc[bi].geometry.contains(pt):
                jordan_mask[li, lj]   = True
                basin_id_grid[li, lj] = bi
                break

elapsed = _time.time() - t0
n_inside  = int(jordan_mask.sum())
n_outside = n_lat * n_lon - n_inside

print(f"  Done ({elapsed:.0f}s)")
print(f"  Inside Jordan  : {n_inside:,} cells")
print(f"  Outside Jordan : {n_outside:,} cells")
print()
print("Grid cells per basin:")
for bi, bname in enumerate(jordan_basin_names):
    n = int(np.sum(basin_id_grid == bi))
    print(f"  {bname:<32} : {n:>4} cells")


Grid      : 85 lat × 75 lon = 6,375 cells
Full axis : 40,177 steps  (1961-01-01 → 2070-12-31)

Building Jordan mask and basin_id grid ...
  Done (3s)
  Inside Jordan  : 840 cells
  Outside Jordan : 5,535 cells

Grid cells per basin:
  HAMMAD                           :  176 cells
  YARMOUK (JORDAN)                 :   11 cells
  J.VALLEY-YARMOUK TRIANGLE        :    0 cells
  JORDAN VALLY (JORDAN)            :    5 cells
  N.R.S.W                          :   11 cells
  AZRAQ (JORDAN)                   :  113 cells
  AMMAN ZARQA (JORDAN)             :   33 cells
  S.R.S.W                          :    6 cells
  MUJIB                            :   61 cells
  D.S.R.S.W                        :   15 cells
  W. ARABA NORTH                   :   27 cells
  HASA                             :   22 cells
  JAFER                            :  116 cells
  WADI ARABA SOUTH                 :   55 cells
  QA DISI & SOUTHERN DESERT        :   36 cells
  SARHAN                           :  153 cells

## 5. Prepare basin_id Encoding

The `basin_id` variable uses the same integer encoding as Notebook 05, making the 3-model and 6-model ensemble NC files directly interchangeable in analysis scripts. Value −1 = outside Jordan.


In [5]:
basin_id_labels = "; ".join(
    f"{bi}={bname}" for bi, bname in enumerate(jordan_basin_names)
)

print("basin_id encoding (same as Notebook 05 / 3-model ensemble):")
for bi, bname in enumerate(jordan_basin_names):
    n = int(np.sum(basin_id_grid == bi))
    print(f"  {bi:>2} = {bname:<32}  cells={n:>4}")
print(f"  -1 = outside Jordan domain  (cells={n_outside:,})")


basin_id encoding (same as Notebook 05 / 3-model ensemble):
   0 = HAMMAD                            cells= 176
   1 = YARMOUK (JORDAN)                  cells=  11
   2 = J.VALLEY-YARMOUK TRIANGLE         cells=   0
   3 = JORDAN VALLY (JORDAN)             cells=   5
   4 = N.R.S.W                           cells=  11
   5 = AZRAQ (JORDAN)                    cells= 113
   6 = AMMAN ZARQA (JORDAN)              cells=  33
   7 = S.R.S.W                           cells=   6
   8 = MUJIB                             cells=  61
   9 = D.S.R.S.W                         cells=  15
  10 = W. ARABA NORTH                    cells=  27
  11 = HASA                              cells=  22
  12 = JAFER                             cells= 116
  13 = WADI ARABA SOUTH                  cells=  55
  14 = QA DISI & SOUTHERN DESERT         cells=  36
  15 = SARHAN                            cells= 153
  -1 = outside Jordan domain  (cells=5,535)


## 6. Generate the Six Unified Jordan-Wide NetCDF Files

**Processing strategy:**  
For each time period, models are accumulated sequentially (one at a time) to keep peak RAM proportional to one model-period array:

1. Allocate a zero accumulator of shape `(n_time, n_lat, n_lon)`  
2. For each model, load only the period time slice and add to accumulator  
3. Divide by 6 for Jordan cells; set non-Jordan cells to `NaN`  
4. Write to compressed NetCDF4 with `basin_id` as a static coordinate variable

**Filename convention:**  
`Jordan_6model_ensemble_ssp245_<PERIOD_LABEL>.nc`


In [6]:
all_times_pd = pd.DatetimeIndex(all_times)

for period_label, y_start, y_end in PERIODS:
    out_filename = f"Jordan_6model_ensemble_ssp245_{period_label}.nc"
    out_path     = OUTPUT_DIR / out_filename

    if out_path.exists():
        print(f"[SKIP] {out_filename} already exists.")
        continue

    # Time indices for this period
    time_mask    = (all_times_pd.year >= y_start) & (all_times_pd.year <= y_end)
    time_idxs    = np.where(time_mask)[0]
    period_times = all_times[time_idxs]
    n_t          = len(time_idxs)

    print(f"[PROCESSING] {out_filename}")
    print(f"  Period  : {y_start}–{y_end}  ({n_t:,} time steps)")
    t0 = _time.time()

    # Allocate accumulator (float64 for precision during summation)
    ens_sum = np.zeros((n_t, n_lat, n_lon), dtype=np.float64)

    for model in MODELS:
        nc_path = MODEL_DIR / f"{model}.nc"
        with xr.open_dataset(nc_path) as ds_m:
            pr_model = ds_m[PR_VAR].isel(time=time_idxs).values  # (n_t, n_lat, n_lon)
        ens_sum += pr_model.astype(np.float64)
        del pr_model
        print(f"  {model:<22} accumulated")

    # Divide by 6; mask outside Jordan
    ens_pr = np.where(
        jordan_mask[np.newaxis, :, :],
        ens_sum / N_MODELS,
        np.nan
    ).astype(np.float32)
    del ens_sum

    # ── Build xarray Dataset ──────────────────────────────────────────────────
    pr_da = xr.DataArray(
        ens_pr,
        dims=["time", "lat", "lon"],
        coords={"time": period_times, "lat": lats, "lon": lons},
        name=PR_VAR,
        attrs={
            "standard_name" : "precipitation_flux",
            "long_name"     : "6-Model Equal-Weight Ensemble Mean Precipitation",
            "units"         : "mm/day",
            "cell_methods"  : "time: mean",
            "ensemble_size" : 6,
            "ensemble_models": " | ".join(MODELS),
            "comment"       : ("Arithmetic mean of all 6 RICCAR CMIP6 models "
                               "at every Jordan domain grid cell. "
                               "Grid cells outside Jordan are NaN."),
        }
    )

    basin_id_da = xr.DataArray(
        basin_id_grid,
        dims=["lat", "lon"],
        coords={"lat": lats, "lon": lons},
        name="basin_id",
        attrs={
            "long_name"      : "Jordan hydrological basin index",
            "flag_meanings"  : " ".join(
                [b.replace(" ", "_").replace(".", "_")
                  .replace("(", "").replace(")", "")
                 for b in jordan_basin_names]
            ) + " outside_jordan",
            "basin_id_labels": basin_id_labels,
            "note"           : "-1 = outside Jordan domain",
        }
    )

    model_list_str = " | ".join(MODELS)

    ds_out = xr.Dataset(
        {PR_VAR: pr_da, "basin_id": basin_id_da},
        attrs={
            "title"              : (f"Jordan-Wide 6-Model Equal-Weight Ensemble "
                                    f"Precipitation {y_start}-{y_end}"),
            "institution"        : "Jordan University of Science and Technology",
            "scenario"           : "SSP2-4.5",
            "period"             : f"{y_start}-{y_end}",
            "ensemble_approach"  : "Equal-weight arithmetic mean of all 6 models",
            "ensemble_models"    : model_list_str,
            "domain"             : "Jordan  (0.1deg x 0.1deg)",
            "original_data"      : ("RICCAR SMHI-HCLIM-ALADIN-38, "
                                    "bias-corrected via SMHI-MIdAS021"),
            "Conventions"        : "CF-1.7",
            "time_coverage_start": str(pd.Timestamp(period_times[0]).date()),
            "time_coverage_end"  : str(pd.Timestamp(period_times[-1]).date()),
            "history"            : (
                f"Created {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
                f"6-model equal-weight ensemble following Hussien et al. Eq.(8)"
            ),
        }
    )

    ds_out["lat"].attrs  = {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}
    ds_out["lon"].attrs  = {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}
    ds_out["time"].attrs = {"long_name": "time"}

    encoding = {
        PR_VAR: {
            "dtype": "float32", "zlib": True, "complevel": 4,
            "_FillValue": np.float32(1e+20),
        },
        "basin_id": {
            "dtype": "int8", "zlib": True, "complevel": 4,
            "_FillValue": np.int8(-1),
        },
    }
    ds_out.to_netcdf(out_path, encoding=encoding, format="NETCDF4")
    del ens_pr, ds_out

    elapsed = _time.time() - t0
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved   : {out_filename}  ({size_mb:.1f} MB)  [{elapsed:.0f}s]")
    print()

print("=" * 65)
print("ALL SIX 6-MODEL ENSEMBLE FILES COMPLETE")
print("=" * 65)


[PROCESSING] Jordan_6model_ensemble_ssp245_1961_2070.nc
  Period  : 1961–2070  (40,177 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved   : Jordan_6model_ensemble_ssp245_1961_2070.nc  (98.4 MB)  [21s]

[PROCESSING] Jordan_6model_ensemble_ssp245_1995_2014.nc
  Period  : 1995–2014  (7,305 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved   : Jordan_6model_ensemble_ssp245_1995_2014.nc  (19.6 MB)  [4s]

[PROCESSING] Jordan_6model_ensemble_ssp245_2015_2020.nc
  Period  : 2015–2020  (2,192 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  EC-Earth3-Veg          accumulated
  I

## 7. Verify Output Files

In [7]:
nc_files = sorted(OUTPUT_DIR.glob("Jordan_6model_ensemble_ssp245_*.nc"))
print(f"6-model ensemble NC files : {len(nc_files)}")
print()

total_mb = 0
for nc_f in nc_files:
    with xr.open_dataset(nc_f) as ds_v:
        pr  = ds_v[PR_VAR]
        n_t = len(ds_v["time"])
        t0  = pd.Timestamp(ds_v["time"].values[0]).date()
        t1  = pd.Timestamp(ds_v["time"].values[-1]).date()
        bid = ds_v["basin_id"].values
        mb  = nc_f.stat().st_size / 1024 / 1024
        total_mb += mb
        print(f"  {nc_f.name}")
        print(f"    Period   : {t0} → {t1}  ({n_t:,} steps)")
        print(f"    pr range : {float(pr.min()):.4f} – {float(pr.max()):.4f} mm/day")
        print(f"    Inside JO: {int(np.sum(bid >= 0)):,} cells | "
              f"Outside: {int(np.sum(bid < 0)):,} cells")
        print(f"    Size     : {mb:.1f} MB")
        print()

print(f"Total output size : {total_mb:.1f} MB")


6-model ensemble NC files : 6

  Jordan_6model_ensemble_ssp245_1961_2070.nc
    Period   : 1961-01-01 → 2070-12-31  (40,177 steps)
    pr range : 0.0000 – 44.2018 mm/day
    Inside JO: 840 cells | Outside: 0 cells
    Size     : 98.4 MB

  Jordan_6model_ensemble_ssp245_1995_2014.nc
    Period   : 1995-01-01 → 2014-12-31  (7,305 steps)
    pr range : 0.0000 – 34.1039 mm/day
    Inside JO: 840 cells | Outside: 0 cells
    Size     : 19.6 MB

  Jordan_6model_ensemble_ssp245_2015_2020.nc
    Period   : 2015-01-01 → 2020-12-31  (2,192 steps)
    pr range : 0.0000 – 35.6241 mm/day
    Inside JO: 840 cells | Outside: 0 cells
    Size     : 6.0 MB

  Jordan_6model_ensemble_ssp245_2021_2040.nc
    Period   : 2021-01-01 → 2040-12-31  (7,305 steps)
    pr range : 0.0000 – 44.2018 mm/day
    Inside JO: 840 cells | Outside: 0 cells
    Size     : 19.7 MB

  Jordan_6model_ensemble_ssp245_2041_2060.nc
    Period   : 2041-01-01 → 2060-12-31  (7,305 steps)
    pr range : 0.0000 – 25.9771 mm/day
    Ins

## 8. Output Summary

In [8]:
print("OUTPUT FILES")
print("=" * 75)
total_mb = 0
for f in sorted(OUTPUT_DIR.glob("Jordan_6model_ensemble_ssp245_*.nc")):
    mb = f.stat().st_size / 1024 / 1024
    total_mb += mb
    print(f"  {f.name:<55}  {mb:>7.1f} MB")
print(f"  {'TOTAL':55}  {total_mb:>7.1f} MB")
print()
print("Models contributing (equal weight at every Jordan grid cell):")
for m in MODELS:
    print(f"  {m}")
print()
print("Next step → run Notebook 09_6model_Extraction_Aggregation.ipynb")


OUTPUT FILES
  Jordan_6model_ensemble_ssp245_1961_2070.nc                  98.4 MB
  Jordan_6model_ensemble_ssp245_1995_2014.nc                  19.6 MB
  Jordan_6model_ensemble_ssp245_2015_2020.nc                   6.0 MB
  Jordan_6model_ensemble_ssp245_2021_2040.nc                  19.7 MB
  Jordan_6model_ensemble_ssp245_2041_2060.nc                  19.7 MB
  Jordan_6model_ensemble_ssp245_2061_2070.nc                   9.9 MB
  TOTAL                                                      173.3 MB

Models contributing (equal weight at every Jordan grid cell):
  CMCC-CM2-SR5
  CNRM-ESM2-1
  EC-Earth3-Veg
  IPSL-CM6A-LR
  MPI-ESM1-2-LR
  NorESM2-MM

Next step → run Notebook 09_6model_Extraction_Aggregation.ipynb
